# MediFlow Hair — EfficientNet-B0 부분 미세조정

이 노트북은 기존 **Augmented 최고 모델**에서 이어서 학습합니다. Google Drive의 `hair_processed` 데이터와 `hair_dataset_results` 결과를 사용하며 기존 파일은 수정하거나 삭제하지 않습니다.

- 연구 가설: EfficientNet-B0 뒷부분을 낮은 학습률로 추가 학습하면 두피 영상의 특징을 더 잘 배울 수 있다.
- 고정 조건: 데이터 분할, 클래스 순서, 224×224 입력, Batch 32, Seed 42, Augmented Train 및 원본 Validation/Test.
- 변경 조건: 동결된 Backbone의 마지막 30개 계층 중 Batch Normalization을 제외한 계층을 학습 가능하게 한다.
- 필수 동반 변경: 기존 특징이 크게 손상되지 않도록 학습률을 `1e-5`로 낮추고 10 Epoch를 추가 학습한다.
- 모델 선택: Validation Accuracy가 가장 높은 체크포인트. Test는 선택 완료 후 한 번만 평가한다.

> 이 결과는 의료 진단이 아니라 연구 및 스크리닝 보조를 위한 분류 결과입니다. 예측 점수를 실제 정답 확률로 해석하지 않습니다.


## 1. Colab과 Google Drive 준비

Colab 메뉴에서 GPU 런타임을 선택한 뒤 아래 셀부터 순서대로 실행합니다. Drive 연결 창이 나타나면 본인의 Drive 계정을 승인합니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install seaborn scikit-learn

import hashlib
import json
import os
import platform
import random
import shutil
import zipfile
from datetime import datetime
from pathlib import Path
from PIL import Image

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

print('Python:', platform.python_version())
print('TensorFlow:', tf.__version__)
print('Keras:', keras.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
if not tf.config.list_physical_devices('GPU'):
    raise RuntimeError('GPU가 없습니다. Colab의 런타임 유형을 GPU로 변경하세요.')


## 2. 실험 설정

화면에서 확인한 Drive 최상위 파일명과 기존 노트북의 하위 폴더 경로를 모두 자동 탐색합니다. 자동 탐색이 실패할 때만 `DATA_ZIP_OVERRIDE` 또는 `BASELINE_OVERRIDE`에 정확한 경로를 입력합니다.


In [ ]:
MY_DRIVE = Path('/content/drive/MyDrive')
DATA_ZIP_OVERRIDE = ''
BASELINE_OVERRIDE = ''  # best_model.keras, 결과 폴더 또는 결과 ZIP 경로

CLASS_NAMES = ['모낭사이홍반', '미세각질', '비듬', '탈모', '피지과다']
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
UNFREEZE_LAST_N = 30
FINE_TUNE_EPOCHS = 10
FINE_TUNE_LEARNING_RATE = 1e-5
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
LOCAL_ROOT = Path('/content') / f'mediflow_hair_partial_finetune_{RUN_ID}'
LOCAL_DATA_ZIP = LOCAL_ROOT / 'hair_processed.zip'
EXTRACT_ROOT = LOCAL_ROOT / 'dataset'
LOCAL_RESULT_DIR = LOCAL_ROOT / 'result'
DRIVE_RESULT_DIR = MY_DRIVE / 'mediflow_experiments' / 'hair' / f'partial_finetune_{RUN_ID}'
DRIVE_AUDIT_DIR = MY_DRIVE / 'mediflow_experiments' / 'hair' / f'data_audit_{RUN_ID}'
AUTOTUNE = tf.data.AUTOTUNE

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
LOCAL_ROOT.mkdir(parents=True, exist_ok=False)
EXTRACT_ROOT.mkdir(parents=True)
LOCAL_RESULT_DIR.mkdir(parents=True)
print('실험 ID:', RUN_ID)
print('Drive 결과 저장 위치:', DRIVE_RESULT_DIR)


## 3. Drive 파일 찾기와 데이터 압축 해제

데이터 ZIP은 Colab 임시 공간으로 복사해 학습 속도를 확보합니다. 복사하면서 SHA-256 식별값을 계산하여 어떤 데이터 파일을 사용했는지 기록합니다.


In [ ]:
def choose_path(override, direct_candidates, search_prefix, require_zip=False):
    if override:
        candidates = [Path(override)]
    else:
        candidates = [p for p in direct_candidates if p.exists()]
        if not candidates:
            candidates = [p for p in MY_DRIVE.rglob(f'{search_prefix}*') if p.is_file()]
    if require_zip:
        candidates = [p for p in candidates if zipfile.is_zipfile(p)]
    if not candidates:
        raise FileNotFoundError(f'Drive에서 {search_prefix} 파일을 찾지 못했습니다. 설정 셀의 경로를 입력하세요.')
    if len(candidates) > 1:
        print('후보가 여러 개여서 첫 번째 파일을 사용합니다:')
        for candidate in candidates:
            print(' -', candidate)
    return candidates[0]

data_candidates = [
    MY_DRIVE / 'hair_processed.zip',
    MY_DRIVE / 'hair_processed',
    MY_DRIVE / 'hair_dataset' / 'hair_processed.zip',
]
DATA_ZIP_PATH = choose_path(DATA_ZIP_OVERRIDE, data_candidates, 'hair_processed', require_zip=True)
print('사용할 데이터 ZIP:', DATA_ZIP_PATH)

def copy_with_sha256(source, destination):
    digest = hashlib.sha256()
    with source.open('rb') as src, destination.open('wb') as dst:
        while True:
            chunk = src.read(8 * 1024 * 1024)
            if not chunk:
                break
            digest.update(chunk)
            dst.write(chunk)
    return digest.hexdigest()

DATA_ZIP_SHA256 = copy_with_sha256(DATA_ZIP_PATH, LOCAL_DATA_ZIP)
print('데이터 ZIP SHA-256:', DATA_ZIP_SHA256)

def safe_extract(zip_path, destination):
    destination = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if destination not in target.parents and target != destination:
                raise ValueError(f'안전하지 않은 ZIP 경로: {member.filename}')
        archive.extractall(destination)

safe_extract(LOCAL_DATA_ZIP, EXTRACT_ROOT)
print('압축 해제 완료:', EXTRACT_ROOT)


## 4. 데이터 구조와 분할 중복 검사

파일 내용이 완전히 같은 이미지가 Original Train/Validation/Test 사이에 있는지 확인하고, Augmented의 Validation/Test가 Original과 같은지 확인합니다. 사람·병변·촬영 세션 식별 정보는 현재 확보되지 않았으므로 그 수준의 누수는 이 검사로 확인할 수 없습니다.


In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def find_dataset_dir(name):
    matches = []
    for candidate in EXTRACT_ROOT.rglob(name):
        if candidate.is_dir() and all((candidate / split).is_dir() for split in ('train', 'val', 'test')):
            matches.append(candidate)
    if len(matches) != 1:
        raise ValueError(f'{name} 데이터 폴더를 하나로 확정할 수 없습니다: {matches}')
    return matches[0]

ORIGINAL_DIR = find_dataset_dir('original')
AUGMENTED_DIR = find_dataset_dir('augmented')
print('Original:', ORIGINAL_DIR)
print('Augmented:', AUGMENTED_DIR)

def image_files(folder):
    return sorted(p for p in folder.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)

def verify_structure(root, label):
    found_classes = sorted(p.name for p in (root / 'train').iterdir() if p.is_dir())
    if found_classes != sorted(CLASS_NAMES):
        raise ValueError(f'{label} 클래스가 계약과 다릅니다: {found_classes}')
    counts = {}
    for split in ('train', 'val', 'test'):
        counts[split] = {}
        for class_name in CLASS_NAMES:
            count = len(image_files(root / split / class_name))
            if count == 0:
                raise ValueError(f'{label}/{split}/{class_name} 이미지가 없습니다.')
            counts[split][class_name] = count
    return counts

original_counts = verify_structure(ORIGINAL_DIR, 'Original')
augmented_counts = verify_structure(AUGMENTED_DIR, 'Augmented')
display(pd.DataFrame(original_counts).rename_axis('class'))
display(pd.DataFrame(augmented_counts).rename_axis('class'))

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def split_hash_groups(root, split):
    groups = {}
    for path in image_files(root / split):
        groups.setdefault(sha256(path), []).append(path)
    return groups

original_hashes = {split: split_hash_groups(ORIGINAL_DIR, split) for split in ('train', 'val', 'test')}
split_pairs = [('train', 'val'), ('train', 'test'), ('val', 'test')]
overlaps = {}
duplicate_rows = []
duplicate_examples = []
for left_split, right_split in split_pairs:
    pair_name = f'{left_split}_{right_split}'
    shared_hashes = sorted(set(original_hashes[left_split]) & set(original_hashes[right_split]))
    overlaps[pair_name] = shared_hashes
    for digest in shared_hashes:
        left_paths = original_hashes[left_split][digest]
        right_paths = original_hashes[right_split][digest]
        duplicate_rows.append({
            'comparison': pair_name,
            'sha256': digest,
            'left_path': str(left_paths[0].relative_to(ORIGINAL_DIR)),
            'right_path': str(right_paths[0].relative_to(ORIGINAL_DIR)),
            'left_copy_count': len(left_paths),
            'right_copy_count': len(right_paths),
        })
        if len(duplicate_examples) < 5:
            duplicate_examples.append((pair_name, digest, left_paths[0], right_paths[0]))

if any(overlaps.values()):
    duplicate_df = pd.DataFrame(duplicate_rows)
    print('분할 사이에서 파일 내용이 완전히 같은 이미지를 발견했습니다.')
    print({name: len(values) for name, values in overlaps.items()})
    display(duplicate_df.head(30))

    DRIVE_AUDIT_DIR.mkdir(parents=True, exist_ok=False)
    report_path = DRIVE_AUDIT_DIR / 'exact_duplicate_report.csv'
    duplicate_df.to_csv(report_path, index=False, encoding='utf-8-sig')

    figure, axes = plt.subplots(len(duplicate_examples), 2, figsize=(12, 4 * len(duplicate_examples)))
    if len(duplicate_examples) == 1:
        axes = np.asarray([axes])
    for row_index, (pair_name, digest, left_path, right_path) in enumerate(duplicate_examples):
        with Image.open(left_path) as left_image:
            axes[row_index, 0].imshow(left_image.convert('RGB'))
        with Image.open(right_path) as right_image:
            axes[row_index, 1].imshow(right_image.convert('RGB'))
        axes[row_index, 0].set_title(str(left_path.relative_to(ORIGINAL_DIR)))
        axes[row_index, 1].set_title(str(right_path.relative_to(ORIGINAL_DIR)))
        axes[row_index, 0].axis('off')
        axes[row_index, 1].axis('off')
        print(f'예시 {row_index + 1}: {pair_name}, SHA-256 {digest}')
    plt.tight_layout()
    example_path = DRIVE_AUDIT_DIR / 'exact_duplicate_examples.png'
    plt.savefig(example_path, dpi=160, bbox_inches='tight')
    plt.show()

    summary = {name: len(values) for name, values in overlaps.items()}
    (DRIVE_AUDIT_DIR / 'duplicate_summary.json').write_text(
        json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    print('중복 검사 결과 저장:', DRIVE_AUDIT_DIR)
    raise ValueError('분할 누수가 있어 학습을 중단했습니다. 위 표와 사진을 확인하세요.')

print('Original 분할 간 완전 동일 파일: 없음')

for split in ('val', 'test'):
    aug_hashes = set(split_hash_groups(AUGMENTED_DIR, split))
    ori_hashes = set(original_hashes[split])
    if aug_hashes != ori_hashes:
        raise ValueError(f'Augmented/{split}와 Original/{split} 이미지 내용이 다릅니다.')
print('Augmented Validation/Test가 Original과 동일함을 확인했습니다.')
print('한계: 사람·병변·촬영 세션 식별 정보가 없어 해당 단위의 누수는 확인하지 못했습니다.')


## 5. 기존 Augmented 최고 모델 찾기

기존 결과 폴더 또는 결과 ZIP에서 `augmented/best_model.keras`를 찾습니다. 기존 결과는 읽기만 하며 삭제하지 않습니다.


In [ ]:
baseline_direct_candidates = [
    MY_DRIVE / 'hair_dataset_results' / 'augmented' / 'best_model.keras',
    MY_DRIVE / 'hair_dataset' / 'hair_model_results' / 'augmented' / 'best_model.keras',
]

def find_augmented_model(root):
    candidates = [p for p in root.rglob('best_model.keras') if p.parent.name.lower() == 'augmented']
    if not candidates:
        raise FileNotFoundError(f'augmented/best_model.keras를 찾지 못했습니다: {root}')
    return candidates[0]

if BASELINE_OVERRIDE:
    baseline_source = Path(BASELINE_OVERRIDE)
elif any(p.is_file() for p in baseline_direct_candidates):
    baseline_source = next(p for p in baseline_direct_candidates if p.is_file())
else:
    result_candidates = [
        MY_DRIVE / 'hair_dataset_results.zip',
        MY_DRIVE / 'hair_dataset_results',
        MY_DRIVE / 'hair_dataset' / 'hair_dataset_results.zip',
    ]
    baseline_source = choose_path('', result_candidates, 'hair_dataset_results', require_zip=False)

if baseline_source.is_file() and baseline_source.suffix.lower() == '.keras':
    BASELINE_MODEL_PATH = baseline_source
elif baseline_source.is_dir():
    BASELINE_MODEL_PATH = find_augmented_model(baseline_source)
elif baseline_source.is_file() and zipfile.is_zipfile(baseline_source):
    local_results = LOCAL_ROOT / 'baseline_results'
    local_results.mkdir()
    safe_extract(baseline_source, local_results)
    BASELINE_MODEL_PATH = find_augmented_model(local_results)
else:
    raise FileNotFoundError(f'기존 결과를 읽을 수 없습니다: {baseline_source}')

print('기준 모델:', BASELINE_MODEL_PATH)


## 6. 학습 데이터와 미세조정 모델 구성

EfficientNet 모델 자체에 `Rescaling(1/255)`이 포함되어 있으므로 외부 `/255.0` 처리는 추가하지 않습니다. Batch Normalization 계층은 작은 Batch에서 기존 통계가 흔들리지 않도록 계속 고정합니다.


In [ ]:
def make_dataset(split, shuffle):
    dataset = tf.keras.utils.image_dataset_from_directory(
        AUGMENTED_DIR / split,
        labels='inferred',
        label_mode='int',
        class_names=CLASS_NAMES,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        seed=SEED if shuffle else None,
    )
    return dataset.prefetch(AUTOTUNE)

train_ds = make_dataset('train', True)
val_ds = make_dataset('val', False)
test_ds = make_dataset('test', False)

model = keras.models.load_model(BASELINE_MODEL_PATH, compile=False)
if int(model.output_shape[-1]) != len(CLASS_NAMES):
    raise ValueError(f'모델 출력 수가 클래스 수와 다릅니다: {model.output_shape[-1]}')

backbone_candidates = [
    layer for layer in model.layers
    if isinstance(layer, keras.Model) and 'efficientnet' in layer.name.lower()
]
if len(backbone_candidates) != 1:
    raise ValueError(f'EfficientNet Backbone을 하나로 찾지 못했습니다: {[x.name for x in backbone_candidates]}')
backbone = backbone_candidates[0]
backbone.trainable = True
for layer in backbone.layers[:-UNFREEZE_LAST_N]:
    layer.trainable = False
for layer in backbone.layers[-UNFREEZE_LAST_N:]:
    layer.trainable = not isinstance(layer, keras.layers.BatchNormalization)

model.compile(
    optimizer=keras.optimizers.Adam(FINE_TUNE_LEARNING_RATE),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
trainable_backbone_layers = [layer.name for layer in backbone.layers if layer.trainable]
print('Backbone:', backbone.name)
print('학습 가능한 Backbone 계층 수:', len(trainable_backbone_layers))
print(trainable_backbone_layers)
model.summary()


## 7. Validation 기준 미세조정

10 Epoch를 실행하고 Validation Accuracy가 가장 높은 모델만 보관합니다. Early Stopping은 사용하지 않아 실행 간 학습량을 고정합니다.


In [ ]:
BEST_MODEL_PATH = LOCAL_RESULT_DIR / 'best_model.keras'
callbacks = [
    keras.callbacks.ModelCheckpoint(
        BEST_MODEL_PATH, monitor='val_accuracy', mode='max', save_best_only=True, verbose=1
    ),
    keras.callbacks.CSVLogger(LOCAL_RESULT_DIR / 'training_log.csv'),
]
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINE_TUNE_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)
best_epoch = int(np.argmax(history.history['val_accuracy'])) + 1
best_val_accuracy = float(max(history.history['val_accuracy']))
print('Best Epoch:', best_epoch)
print(f'Best Validation Accuracy: {best_val_accuracy:.4f}')


## 8. 선택 완료 후 Test 1회 평가

여기서 얻은 Test 결과를 보고 설정을 반복 변경하지 않습니다. 다음 실험은 새 가설과 새 결과 폴더로 분리합니다.


In [ ]:
best_model = keras.models.load_model(BEST_MODEL_PATH, compile=False)
y_true, y_pred = [], []
for images, labels in test_ds:
    probabilities = best_model.predict(images, verbose=0)
    y_true.extend(labels.numpy().tolist())
    y_pred.extend(np.argmax(probabilities, axis=1).tolist())
y_true = np.asarray(y_true)
y_pred = np.asarray(y_pred)
test_accuracy = float(accuracy_score(y_true, y_pred))
macro_f1 = float(f1_score(y_true, y_pred, average='macro'))
report = classification_report(
    y_true, y_pred, labels=list(range(len(CLASS_NAMES))), target_names=CLASS_NAMES,
    output_dict=True, zero_division=0
)
print(classification_report(
    y_true, y_pred, labels=list(range(len(CLASS_NAMES))), target_names=CLASS_NAMES,
    digits=6, zero_division=0
))
print(f'새 Test Accuracy: {test_accuracy:.4f}')
print(f'새 Macro F1: {macro_f1:.4f}')

cm = confusion_matrix(y_true, y_pred, labels=list(range(len(CLASS_NAMES))))
plt.figure(figsize=(9, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Hair Partial Fine-tuning Confusion Matrix')
plt.tight_layout()
plt.savefig(LOCAL_RESULT_DIR / 'confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.show()


## 9. 기록 저장과 Drive 백업

기존 보고 결과와 이번에 새로 측정한 결과를 구분해 저장합니다. 기존 Drive 결과 폴더를 지우거나 덮어쓰지 않습니다.


In [ ]:
history_json = {key: [float(value) for value in values] for key, values in history.history.items()}
config = {
    'experiment': 'hair_partial_finetuning',
    'hypothesis': 'EfficientNet-B0 뒷부분 미세조정이 두피 분류 성능을 개선한다.',
    'data_zip_path': str(DATA_ZIP_PATH),
    'data_zip_sha256': DATA_ZIP_SHA256,
    'split_counts_original': original_counts,
    'split_counts_augmented': augmented_counts,
    'class_names': CLASS_NAMES,
    'baseline_model_path': str(BASELINE_MODEL_PATH),
    'local_repository_commit_at_notebook_creation': 'b5faa6d937229a49e9d62541a30e39f3b75a3c77',
    'notebook_state_at_creation': 'uncommitted',
    'image_size': list(IMAGE_SIZE),
    'batch_size': BATCH_SIZE,
    'seed': SEED,
    'unfreeze_last_n': UNFREEZE_LAST_N,
    'batch_normalization_frozen': True,
    'fine_tune_epochs': FINE_TUNE_EPOCHS,
    'fine_tune_learning_rate': FINE_TUNE_LEARNING_RATE,
    'selection_metric': 'val_accuracy',
    'best_epoch': best_epoch,
    'best_val_accuracy': best_val_accuracy,
    'test_accuracy': test_accuracy,
    'macro_f1': macro_f1,
    'environment': {
        'python': platform.python_version(),
        'tensorflow': tf.__version__,
        'keras': keras.__version__,
        'numpy': np.__version__,
    },
    'leakage_check': {
        'exact_content_duplicates_across_original_splits': False,
        'augmented_val_test_equal_original': True,
        'person_lesion_session_check': '확인 불가: 식별 정보 미확보',
    },
}
comparison = {
    'existing_reported_baseline': {
        'source': 'results/hair/augmented 기존 로컬 보고서',
        'test_accuracy': 0.6910344827586207,
        'macro_f1': 0.6894861696407794,
    },
    'newly_measured_partial_finetuning': {
        'best_val_accuracy': best_val_accuracy,
        'test_accuracy': test_accuracy,
        'macro_f1': macro_f1,
    },
    'expected_result': '사전에 수치로 정하지 않음',
}

(LOCAL_RESULT_DIR / 'training_config.json').write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
(LOCAL_RESULT_DIR / 'training_history.json').write_text(json.dumps(history_json, ensure_ascii=False, indent=2), encoding='utf-8')
(LOCAL_RESULT_DIR / 'comparison.json').write_text(json.dumps(comparison, ensure_ascii=False, indent=2), encoding='utf-8')
pd.DataFrame(report).transpose().to_csv(LOCAL_RESULT_DIR / 'classification_report.csv', encoding='utf-8-sig')
pd.DataFrame(history.history).to_csv(LOCAL_RESULT_DIR / 'training_history.csv', index_label='epoch_index')

if DRIVE_RESULT_DIR.exists():
    raise FileExistsError(f'결과 폴더가 이미 있습니다. 덮어쓰지 않습니다: {DRIVE_RESULT_DIR}')
DRIVE_RESULT_DIR.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(LOCAL_RESULT_DIR, DRIVE_RESULT_DIR)
print('Drive 저장 완료:', DRIVE_RESULT_DIR)
for path in sorted(DRIVE_RESULT_DIR.iterdir()):
    print(' -', path.name)


## 실행 완료 후 확인할 것

Drive의 `mediflow_experiments/hair/partial_finetune_실행시각/` 폴더를 확인합니다. `comparison.json`에는 기존 보고 결과와 새 측정 결과가 구분되어 있습니다. 새 결과가 좋아도 같은 Test를 보며 설정을 다시 조정하지 말고, 다음 실험은 Validation 기준의 별도 가설로 설계합니다.
